In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import scipy.sparse as sp
from pathlib import Path

In [2]:
BASE = Path("/project/Wellcome_Discovery/datashare/lturiano/GENESIS/data")

organ = "heart"            # a string
organ_path = BASE / organ  # use / to join
print(organ_path)          # prints a Path

/project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/heart


## SN+SC: Combined single cell and single nuclei RNA-Seq data - Heart Global
Link: https://cellxgene.cziscience.com/collections/3116d060-0a8e-4767-99bb-e866badea1ed

In [3]:
# Load the .h5ad file
adata = sc.read_h5ad(BASE / organ / f"{organ}_combined.h5ad")

In [5]:
adata.obs["cell_type"].value_counts()

cell_type
regular ventricular cardiac myocyte     190710
fibroblast                              138055
endothelial cell                        131505
mural cell                              104593
myeloid cell                             51426
regular atrial cardiac myocyte           45911
lymphocyte                               24922
neural cell                               6622
adipocyte                                 6347
mast cell                                 1853
endothelial cell of lymphatic vessel      1295
mesothelial cell                          1057
Name: count, dtype: int64

In [6]:
cell_type = [dict_ct[ct] for ct in adata.obs['cell_type']]
adata.obs['cell_type'] = cell_type
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

In [8]:
adata.obs['cell_type'].value_counts()

cell_type
regular ventricular cardiac myocyte     190710
fibroblast                              138055
endothelial cell                        131505
mural cell                              104593
myeloid cell                             51426
regular atrial cardiac myocyte           45911
lymphocyte                               24922
neural cell                               6622
adipocyte                                 6347
mast cell                                 1853
endothelial cell of lymphatic vessel      1295
mesothelial cell                          1057
Name: count, dtype: int64

In [16]:
ct_to_remove = ['mast cell', 'mesothelial cell', 'endothelial cell of lymphatic vessel',
                'regular atrial cardiac myocyte', 'regular ventricular cardiac myocyte',
               'neural cell', 'adipocyte']

adata = adata[~adata.obs['cell_type'].isin(ct_to_remove)]

In [17]:
mask_sn = adata.obs["suspension_type"] == "nucleus"
mask_sc = adata.obs["suspension_type"] == "cell"

adata_sn = adata[mask_sn]
adata_sc = adata[mask_sc]

In [18]:
# Save
adata_sn.write_h5ad(BASE / organ / f"{organ}_sn.h5ad")
adata_sc.write_h5ad(BASE / organ / f"{organ}_sc.h5ad")

## Load processed data

In [19]:
# Load the .h5ad file
adata_sn = sc.read_h5ad(BASE / organ / f"{organ}_sn.h5ad")
adata_sc = sc.read_h5ad(BASE / organ / f"{organ}_sc.h5ad")

In [20]:
common_cell_types = adata_sn.obs["cell_type"].unique()
common_cell_types

['fibroblast', 'mural cell', 'myeloid cell', 'endothelial cell', 'lymphocyte']
Categories (5, object): ['fibroblast', 'endothelial cell', 'mural cell', 'myeloid cell', 'lymphocyte']

In [21]:
# Create a df table to show me how many cells of each cell type are in the two datasets, just for the intersection cell types
cell_type_counts_sn = adata_sn.obs["cell_type"].value_counts()
cell_type_counts_sc = adata_sc.obs["cell_type"].value_counts()
df_counts = pd.DataFrame({
    "cell_type": list(common_cell_types),
    "count_sn": [cell_type_counts_sn[cell_type] for cell_type in common_cell_types],
    "count_sc": [cell_type_counts_sc[cell_type] for cell_type in common_cell_types]
})

df_counts["min_count"] = df_counts[["count_sn", "count_sc"]].min(axis=1)
df_counts

,cell_type,count_sn,count_sc,min_count
0,fibroblast,134892,3163,3163
1,mural cell,81952,22641,22641
2,myeloid cell,36724,14702,14702
3,endothelial cell,51107,80398,51107
4,lymphocyte,12977,11945,11945


In [22]:
def balance_two_adatas_by_celltype(
    adata_a,
    adata_b,
    celltype_col="cell_type",
    celltypes=None,
    random_state=0,
    force_targets=None,
):
    rng = np.random.default_rng(random_state)

    # pick shared cell types if not provided
    if celltypes is None:
        celltypes = sorted(set(adata_a.obs[celltype_col]).intersection(set(adata_b.obs[celltype_col])))

    counts_a = adata_a.obs[celltype_col].value_counts()
    counts_b = adata_b.obs[celltype_col].value_counts()

    # min per cell type
    min_counts = {}
    for ct in celltypes:
        if ct in counts_a and ct in counts_b:
            min_counts[ct] = int(min(counts_a[ct], counts_b[ct]))

    # override targets if requested (clip to what's available in BOTH datasets)
    if force_targets:
        for ct, k in force_targets.items():
            if ct in min_counts:
                min_counts[ct] = int(min(k, counts_a[ct], counts_b[ct]))

    def subset_to_targets(adata, targets):
        idx_keep = []
        for ct, k in targets.items():
            idx_ct = adata.obs_names[adata.obs[celltype_col] == ct].to_numpy()
            if len(idx_ct) <= k:
                idx_keep.append(idx_ct)
            else:
                idx_keep.append(rng.choice(idx_ct, size=k, replace=False))
        idx_keep = np.concatenate(idx_keep) if len(idx_keep) else np.array([], dtype=object)
        return adata[idx_keep].copy()

    adata_a_bal = subset_to_targets(adata_a, min_counts)
    adata_b_bal = subset_to_targets(adata_b, min_counts)

    summary = pd.DataFrame({
        "cell_type": list(min_counts.keys()),
        "count_a_before": [int(counts_a[ct]) for ct in min_counts.keys()],
        "count_b_before": [int(counts_b[ct]) for ct in min_counts.keys()],
        "target_count":   [int(min_counts[ct]) for ct in min_counts.keys()],
    }).sort_values("cell_type").reset_index(drop=True)

    return adata_a_bal, adata_b_bal, summary

In [23]:
adata_sn_bal, adata_sc_bal, summary = balance_two_adatas_by_celltype(
    adata_sn, adata_sc,
    celltype_col="cell_type",
    celltypes=common_cell_types,
    random_state=42,
    force_targets=None
)

In [24]:
adata_sn_bal.obs["cell_type"].value_counts()

cell_type
endothelial cell    51107
mural cell          22641
myeloid cell        14702
lymphocyte          11945
fibroblast           3163
Name: count, dtype: int64

In [25]:
adata_sc_bal.obs["cell_type"].value_counts()

cell_type
endothelial cell    51107
mural cell          22641
myeloid cell        14702
lymphocyte          11945
fibroblast           3163
Name: count, dtype: int64

In [26]:
X_sn = adata_sn_bal.X.toarray()
X_sc = adata_sc_bal.X.toarray()

obs_sn = pd.DataFrame(adata_sn_bal.obs['cell_type'])
var_sn = pd.DataFrame(adata_sn_bal.var['feature_name'])

obs_sc = pd.DataFrame(adata_sc_bal.obs['cell_type'])
var_sc = pd.DataFrame(adata_sc_bal.var['feature_name'])

In [27]:
adata_sn_final = ad.AnnData(X=X_sn, obs=obs_sn, var=var_sn)
adata_sc_final = ad.AnnData(X=X_sc, obs=obs_sc, var=var_sc)

In [28]:
# Save
adata_sn_final.write_h5ad(BASE / organ / f"{organ}_sn_filt.h5ad", compression="gzip")
adata_sc_final.write_h5ad(BASE / organ / f"{organ}_sc_filt.h5ad", compression="gzip")

## Analysis

In [25]:
# Load the .h5ad file
adata_sn_final = sc.read_h5ad(BASE / organ / f"{organ}_sn_filt.h5ad")
adata_sc_final = sc.read_h5ad(BASE / organ / f"{organ}_sc_filt.h5ad")

In [36]:
marker_genes = {
    "endothelial cell": ["PECAM1", "VWF", "KDR", "CLDN5", "ENG"],
    "fibroblast": ["COL1A1", "COL1A2", "DCN", "LUM", "PDGFRA"],
    "lymphocyte": ["PTPRC", "CD3D", "CD3E", "IL7R", "CD2"],
    "mural_cell": ["RGS5", "PDGFRB", "ACTA2", "TAGLN", "MYH11"],
    "myeloid cell": ["LYZ", "CD68", "CSF1R", "HLA-DRA", "ITGAM"]
}

In [37]:
genes = []

for row in list(marker_genes.values()):
    for item in row:
        if item not in genes:
            genes.append(item)
            
np.random.shuffle(genes)

In [38]:
# sn_genes = list(adata_sn_final.var.feature_name)
sc_genes = list(adata_sn_final.var.feature_name)

In [40]:
genes_not_found = []
for gene in genes:
    if gene not in sc_genes:
        genes_not_found.append()
        
if genes_not_found:
    print(f"Marker genes not present in sc_genes: {genes_not_found}\n")
else:
    print("All the marker genes are present in sc_genes!\n")

All the marker genes are present in sc_genes!

